# 03 — VaR and sVaR Analysis

Calculate Historical Simulation VaR, stressed VaR, and hierarchy-level risk reports using full scenario revaluation.

The key production pattern: **Greeks are aggregated additively, while VaR is recalculated from grouped P&L distributions.**


## 1. Notebook setup


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.6f}".format)

DATA_DIR = PROJECT_ROOT / "data" / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)


## 2. Configuration and inputs


In [3]:
from src.config.settings import RiskSettings
from src.pricing.instruments import MarketContext
from src.data.generate_positions import generate_synthetic_positions
from src.data.generate_structure import generate_structure
from src.data.market_data import enrich_positions_with_market, load_market_data
from src.pricing.greeks import calculate_instrument_risk

settings = RiskSettings(confidence_level=0.99, lookback_days=252, stress_window_days=252)

market = load_market_data(settings.tickers, settings.valuation_date, seed=settings.random_seed, use_yfinance=False)
positions = generate_synthetic_positions(settings.valuation_date)
structure = generate_structure()
enriched = enrich_positions_with_market(positions, structure, market, settings.valuation_date)

ctx = MarketContext(
    valuation_date=pd.Timestamp(settings.valuation_date),
    risk_free_rate=settings.risk_free_rate,
    dividend_yield=settings.dividend_yield,
    vols=settings.vols(),
    fx_foreign_rates={"EURUSD=X": 0.015, "GBPUSD=X": 0.018},
)

instrument_risk = calculate_instrument_risk(enriched, ctx)
instrument_risk.head()


,PositionID,InstrumentType,Ticker,Quantity,Portfolio,Maturity,Strike,OptionType,TradingDesk,Unit,CurrentPrice,MarketValue_Initial,NPV,Delta,Gamma,Vega,Theta,Rho
0,POS001,Stock,AAPL,1000,P1_EqUS,NaT,NaN,NaN,Equity Desk,Trading Unit A,136.500838,136500.838000,136500.838000,1000.000000,0.000000,0.000000,0.000000,0.000000
1,POS002,Stock,GOOG,500,P1_EqUS,NaT,NaN,NaN,Equity Desk,Trading Unit A,41.469261,20734.630500,20734.630500,500.000000,0.000000,0.000000,0.000000,0.000000
2,POS003,FX Forward,EURUSD=X,1000000,P2_FXMaj,2025-07-03,1.080000,NaN,FX Desk,Trading Unit A,1.400782,NaN,320923.511220,996308.201370,0.000000,0.000000,0.000000,0.000000
3,POS004,FX Forward,GBPUSD=X,-500000,P2_FXMaj,2025-10-01,1.250000,NaN,FX Desk,Trading Unit A,0.903911,NaN,170904.541821,-495581.284683,0.000000,0.000000,0.000000,0.000000
4,POS005,European Option,AAPL,50,P3_OptEq,2025-06-03,170.000000,Call,Options Desk,Trading Unit B,136.500838,NaN,0.729729,0.216347,0.057413,0.351699,-0.060195,0.047346


## 3. Historical Simulation VaR

Build log-return scenarios from the lookback window, shock current market prices, and revalue every position under each scenario.


In [4]:
from src.risk.var import build_historical_scenarios, calculate_var

scenarios = build_historical_scenarios(market, settings.lookback_days)
portfolio_var, pnl = calculate_var(
    positions=instrument_risk,
    market_data=market,
    ctx=ctx,
    confidence_level=settings.confidence_level,
    lookback_days=settings.lookback_days,
)

print(f"Portfolio VaR {settings.confidence_level:.0%}: {portfolio_var:,.2f}")
pnl.head()


Traceback (most recent call last):
  File "C:\Program Files\JetBrains\PyCharm 2025.3.3\plugins\python-ce\helpers\pydev\_pydevd_bundle\pydevd_comm.py", line 736, in make_thread_stack_str
    append('file="%s" line="%s">' % (make_valid_xml_value(my_file), lineno))
                                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\JetBrains\PyCharm 2025.3.3\plugins\python-ce\helpers\pydev\_pydevd_bundle\pydevd_xml.py", line 36, in make_valid_xml_value
    return s.replace("&", "&amp;").replace('<', '&lt;').replace('>', '&gt;').replace('"', '&quot;')
           ^^^^^^^^^
AttributeError: 'tuple' object has no attribute 'replace'


KeyboardInterrupt: 

## 4. Stressed scenario selection

The stress window is selected around the worst anchor-ticker return. This is simple but transparent and easy to replace with a governance-approved stress calendar.


In [ ]:
from src.risk.svar import select_stress_scenarios
from src.risk.revaluation import portfolio_pnl_distribution
from src.risk.var import var_from_pnl

stress_scenarios = select_stress_scenarios(market, settings.stress_window_days, anchor_ticker="AAPL")
stress_pnl = portfolio_pnl_distribution(instrument_risk, market, ctx, stress_scenarios)
portfolio_svar = var_from_pnl(stress_pnl["PortfolioPnL"], settings.confidence_level)

print(f"Stress window: {stress_scenarios.index.min().date()} to {stress_scenarios.index.max().date()}")
print(f"Portfolio sVaR {settings.confidence_level:.0%}: {portfolio_svar:,.2f}")


## 5. Desk and unit reports

These reports combine additive Greeks with full-revaluation VaR/sVaR at each hierarchy level.


In [ ]:
from src.aggregation.hierarchy import hierarchy_report

desk_report = hierarchy_report(
    instrument_risk,
    market,
    ctx,
    settings.confidence_level,
    settings.lookback_days,
    settings.stress_window_days,
    group_col="TradingDesk",
)
unit_report = hierarchy_report(
    instrument_risk,
    market,
    ctx,
    settings.confidence_level,
    settings.lookback_days,
    settings.stress_window_days,
    group_col="Unit",
)

display(desk_report.round(6))
display(unit_report.round(6))


## 6. VaR distribution diagnostics

Review the P&L distribution and left tail to verify the Historical Simulation result.


In [ ]:
import matplotlib.pyplot as plt

ax = pnl["PortfolioPnL"].hist(bins=40, figsize=(11, 5))
ax.axvline(portfolio_var, linestyle="--", linewidth=2)
ax.set_title("Historical Simulation Portfolio P&L Distribution")
ax.set_xlabel("Scenario P&L")
ax.set_ylabel("Frequency")
plt.tight_layout()


In [ ]:
tail = pnl[["PortfolioPnL"]].sort_values("PortfolioPnL").head(10)
tail


## 7. Persist VaR outputs


In [ ]:
pnl.to_csv(DATA_DIR / "portfolio_pnl.csv")
stress_pnl.to_csv(DATA_DIR / "stress_portfolio_pnl.csv")
desk_report.to_csv(DATA_DIR / "risk_report_by_desk.csv", index=False)
unit_report.to_csv(DATA_DIR / "risk_report_by_unit.csv", index=False)

print("Wrote VaR and hierarchy outputs to data/processed")


## 8. Production notes

Recommended next extensions:

1. Replace simulated curves with bootstrapped OIS/IBOR curves.
2. Add volatility surface calibration and scenario vol shocks.
3. Persist scenario-level explain by risk factor.
4. Add portfolio-level limit checks and breach flags.
5. Package notebook execution in CI with `nbmake` or `papermill`.
